<a href="https://colab.research.google.com/github/SolJosefinaMercado/ProcesamientoDelHabla/blob/main/TP5_MercadoSol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP5 Chatbots
## Sol Josefina Mercado 38442698


# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Actividades

# TP5

Copio las celdas del TP3 que rehuso


In [ ]:
import pandas as pd

# ID del archivo de Google Drive desde tu enlace
file_id = '1YjBek4zUQNLhTT0citWVogNpo2TsOHBv'
url_descarga = f'https://drive.google.com/uc?export=download&id={file_id}'


df_importado = pd.read_csv(url_descarga)
display(df_importado.head(2))

In [ ]:
# Creo el diccionario de respuestas asociando cada intención REAL del dataset con su respuesta
respuestas = {
    'saludo': '¡Hola! ¿Cómo estás? Bienvenido a nuestro sistema de atención. ¿En qué te puedo ayudar hoy?',
    'info_entrenamiento': 'Ofrecemos entrenamientos adaptados a tus objetivos: musculación, funcional, y entrenamiento de fuerza. Apto para todos los niveles',
    'cupo_horario': 'Los cupos para las clases presenciales y los horarios disponibles se gestionan y reservan mediante nuestra aplicación móvil, el link es (...)',
    'precio_abono': 'Tenemos diferentes planes: el pase libre mensual cuesta $$$ y el pase de 3 veces por semana cuesta $$$.',
    'info_levantamiento_olimpico': 'Las clases de levantamiento olímpico de pesas se enfocan en la técnica del arranque y envión para todos los niveles.',
    'inicio_powerlifting': '¡Por supuesto! Puedes empezar Powerlifting sin experiencia previa. Nos adaptamos a tu nivel inicial y te enseñamos la técnica de sentadilla, banco y peso muerto desde cero.',
    'incripcion' : 'Podes acceder a la app de turnos desde el sigueinte link (...) Accedes con tu numero de DNI una vez inscrito/a'
}

# Guardamos las respuestas en el DataFrame para facilitar el entrenamiento posterior
df_importado['respuesta'] = df_importado['intencion'].map(respuestas)

print("\nVista previa del DataFrame con las respuestas correctamente integradas y sin nulos:")
display(df_importado[['intencion', 'pregunta', 'respuesta']].dropna().head(5))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

## Dataset ampliado de tp3

In [ ]:
preguntas = [
    # intencion: SALUDO
    'Hola! Como estás?',
    'Hola buenas!',
    'buenas como andas',
    'Hola, qué tal?',
    'holaa todo bien?',
    'Buen día!',
    'buenas tardes',
    'hola q tal',
    # intencion: INFO_ENTRENAMIENTO
    'Hola!! Cómo estás? Quisiera info de entrenamiento por favor. Y quisiera saber si hay cupo a las 15hs...Gracias!!',
    'Hola, me pasarías info de los entrenamientos?',
    'buenas, quería info del gym',
    'q tipo de entrenamiento hacen?',
    'Hola! Me interesa entrenar, cómo funciona?',
    'me podes pasar informacion de las clases?',
    'como son los entrenamientos? son personalizados?',
    'quisiera saber mas sobre el gimnasio',
    'info x favor',
    'Hay maquinas para musculacion?',
    'tienen entrenamiento funcional?',
    'qué se puede hacer en el gym?',
    # intencion: CUPO_HORARIO
    'hay lugar a las 15?',
    'tenes cupo a la tarde?',
    'Queda lugar en el turno de las 3 de la tarde?',
    'que horarios tienen disponibles?',
    'a q hora puedo ir a entrenar?',
    'hay turnos libres a la mañana?',
    'en que horarios abren?',
    'tenes horarios a la noche?',
    'que horarios hay disponibles?',
    # intencion: PRECIO_ABONO
    'cual es el precio del abono mensual? Gracias!',
    'cuanto sale la cuota?', 'q valor tiene el mes?',
    'Hola, cuál es el precio de la mensualidad?',
    'cuanto cobras x mes',
    'precio?',
    'cuanto es la cuota mensual',
    'me pasas los precios de los planes?',
    'cuanto sale entrenar 3 veces x semana?',
    # intencion: INFO_OLIMPICO
    'Buenas tardes ,consulta vi en una publicación que dan levantamiento olímpico quería pedirte info sobre esa clase , gracias',
    'dan clases de levantamiento olimpico?',
    'vi que hacen halterofilia, como es la clase?',
    'Hola! Quería info sobre las clases de olímpico',
    'tienen clases de arranque y envion?',
    'vi en insta lo de levantamiento olimpico, me pasas info?',
    'como son las clases de lev olimpico',
    'puedo hacer olimpico sin experiencia?',
    # intencion: INICIO_POWER
    'Hola como estas ? Te cuento hize crosfitt , pero bueno avandone hace unos meses vengo viendo tus videos , me gustaria arrancar con tus clases  powerlifting',
    'quiero empezar powerlifting, como hago?',
    'me gustaria arrancar con powerlifting',
    'Hola! Hice crossfit hace un tiempo y quiero volver a entrenar, me sumo a powerlifting?',
    'como me anoto a las clases de power?',
    'vengo viendo tus videos y quiero empezar a entrenar con vos',
    'nunca hice powerlifting, puedo empezar igual?',
    'hace meses que no entreno, quiero arrancar con fuerza',
    'quiero entrenar sentadilla banco y peso muerto, dan clases?',
    # intencion: INSCRIPCION
    'cómo me inscribo',
    'donde descargo la app',
    'como hago para anotarme?',
    'q tengo que hacer para inscribirme',
    'como me registro en la app del gym?',
    'cual es la app para reservar las clases?',
    'quiero anotarme, que necesito?',
    'tienen link para bajar la aplicacion?']

intenciones = ['saludo', 'saludo', 'saludo', 'saludo', 'saludo', 'saludo', 'saludo', 'saludo',
               'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento', 'info_entrenamiento',
               'cupo_horario','cupo_horario', 'cupo_horario', 'cupo_horario', 'cupo_horario', 'cupo_horario', 'cupo_horario', 'cupo_horario', 'cupo_horario',
               'precio_abono', 'precio_abono', 'precio_abono', 'precio_abono', 'precio_abono', 'precio_abono', 'precio_abono', 'precio_abono', 'precio_abono',
               'info_levantamiento_olimpico', 'info_levantamiento_olimpico', 'info_levantamiento_olimpico', 'info_levantamiento_olimpico', 'info_levantamiento_olimpico', 'info_levantamiento_olimpico', 'info_levantamiento_olimpico', 'info_levantamiento_olimpico',
               'inicio_powerlifting', 'inicio_powerlifting', 'inicio_powerlifting', 'inicio_powerlifting', 'inicio_powerlifting', 'inicio_powerlifting', 'inicio_powerlifting', 'inicio_powerlifting',
               'inscripcion', 'inscripcion', 'inscripcion', 'inscripcion', 'inscripcion', 'inscripcion', 'inscripcion', 'inscripcion']

respuestas = {
    'saludo': '¡Hola! ¿Cómo estás? Bienvenido a nuestro sistema de atención. ¿En qué te puedo ayudar: horarios, cupos, abonos...?',
    'info_entrenamiento': 'Ofrecemos entrenamientos adaptados a tus objetivos: musculación, funcional, y entrenamiento de fuerza. Apto para todos los niveles',
    'cupo_horario': 'Los cupos para las clases presenciales y los horarios disponibles se gestionan y reservan mediante nuestra aplicación móvil, el link es (...) y los horarios disponibles son 8:30,15,16,17,18,19 y 20hs',
    'precio_abono': 'Tenemos diferentes planes: el pase libre mensual cuesta $$$ y el pase de 3 veces por semana cuesta $$$.',
    'info_levantamiento_olimpico': 'Las clases de levantamiento olímpico de pesas se enfocan en la técnica del arranque y envión para todos los niveles.',
    'inicio_powerlifting': '¡Por supuesto! Puedes empezar Powerlifting sin experiencia previa. Nos adaptamos a tu nivel inicial y te enseñamos la técnica de sentadilla, banco y peso muerto desde cero.',
    'inscripcion' : 'Podes acceder a la app de turnos desde el sigueinte link (...) Accedes con tu numero de DNI una vez inscrito/a, dentro está toda la informacion detallada sobre cupos y horarios'
}

df_ampliado = pd.DataFrame({'pregunta': preguntas,
                            'intencion': intenciones})
df_ampliado['respuesta'] = df_ampliado['intencion'].map(respuestas)
df_ampliado.head(5)


# a) Creación del conjunto de datos de evaluación.
Armé un dataset de evaluación con 18 preguntas que no están en el dataset del TP3: 2 por intención, con sinónimos, abreviaturas y errores de tipeo, como escriben los clientes por WhatsApp. Agregué 4 preguntas fuera de dominio para comprobar que el chatbot no invente respuestas sobre temas que no están cargados.

In [ ]:
# @title
# Dataset de evaluación: preguntas NUEVAS, que no están en el dataset del TP3
preguntas_eval = [
  'buenas noches!',
  'holis como va?',
  'que actividades tienen?',
  'hacen funcional o solo pesas?',
  'tenes lugar mañana a las 18?',
  'hasta q hora se puede ir?',
  'cuanto esta el mes?',
  'que valor tiene la mensualidad?',
  'que onda las clases de halterofilia?',
  'enseñan arranque y envion?',
  'nunca levante pesas, puedo hacer powerlifting?',
  'quiero aprender sentadilla y peso muerto',
  'como me doy de alta?',
  'que app tengo que bajar?',
  'tienen estacionamiento?',
  'venden suplementos?',
  'hay vestuarios?',
  'puedo dejar la moto adentro?',
]

intenciones_eval = [
  'saludo', 'saludo',
  'info_entrenamiento', 'info_entrenamiento',
  'cupo_horario', 'cupo_horario',
  'precio_abono', 'precio_abono',
  'info_levantamiento_olimpico', 'info_levantamiento_olimpico',
  'inicio_powerlifting', 'inicio_powerlifting',
  'inscripcion', 'inscripcion',
  'fuera_de_dominio', 'fuera_de_dominio',
  'fuera_de_dominio', 'fuera_de_dominio',
]

df_eval = pd.DataFrame({'pregunta': preguntas_eval, 'intencion': intenciones_eval})

# respuesta esperada: la del diccionario del TP3
df_eval['respuesta_esperada'] = df_eval['intencion'].map(respuestas)
# las fuera de dominio no están en el diccionario (quedan vacías), les pongo el mensaje de "no sé"
df_eval['respuesta_esperada'] = df_eval['respuesta_esperada'].fillna('No tengo esa información')

# control: tiene que imprimir una lista vacía (ninguna pregunta repetida del TP3)
print([p for p in preguntas_eval if p in preguntas])
df_eval.head(5)

# b) Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings. Justifique su elección.

Elegí dos modelos de embeddings de sentence-transformers multilingües, entrenados para detectar paráfrasis.
Ya que uno de los problemas detectados en el TP3, y motivo or el cul utilicé un dataset de preguntas y respuestas agrupado por intenciones, es justamente reconocer cuando un cliente pregunta lo mismo con otras palabras.

Embeding A: `MiniLM`, (384 dimensiones)

Embeding B: `mpnet`, (768 dimensiones)

A diferencia de spaCy en el TP3, que promediaba vectores de palabras, estos modelos generan un vector por oración completa.

Como LLM elegí `Qwen2.5-1.5B-Instruct` porque sigue instrucciones que es lo que necesito para el chatbot, maneja bien el español y corre en la GPU gratuita de Colab y tiene licencia apache 2.0 por lo que no es necesario utilizar tokens.
Otros modelos tenidos en cuenta ocupaban demasiado espacio para la GPU de colab, funcoinaban mejor en idioma inglés, o requerian acceso por tokens.

In [ ]:
!pip install -q chromadb sentence-transformers accelerate

In [ ]:
# @title
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline
import torch

emb_a = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb_b = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# carga el LLM (la primera vez descarga unos 3 GB, tarda un par de minutos)
# cargo el modelo en media precision, con 16 bits ocupa la mitad de memoria en la GPU T4 corre mas rapido
# sin perder calidad
llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               device_map="auto", torch_dtype=torch.float16)

'''
modelo mas chico:
llm = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
               device_map="auto", torch_dtype=torch.float16)
'''

## Prueba, el LLM funciona?
Elegí Qwen2.5-1.5B-Instruct porque es un modelo Instruct, entrenado para seguir instrucciones, que es lo que necesita un chatbot. Es multilingüe y maneja bien el español. Con 1.500 millones de parámetros ocupa unos 3 GB y entra en la GPU gratuita de Colab. Tiene licencia Apache 2.0, así que no requiere pedir acceso ni usar un token. Consideré alternativas como Llama-3.2 y Gemma-2, pero requieren pedir acceso, y modelos más grandes (7B) no entran en la GPU. Como en un esquema RAG el LLM solo redacta a partir de la información recuperada, un modelo chico es suficiente.

In [ ]:
# @title
mensajes = [{"role": "user", "content": "Saludame en una oración breve y dame la bienvenida al gimnasio"}]
salida = llm(mensajes, max_new_tokens=30,do_sample=False)
print(salida[0]["generated_text"][-1]["content"])

## Prueba: Los embeddings seleccionados entienden sinónimos?
Ambos modelos asignan alta similitud a “cuota” y “mensualidad” (0.72 y 0.77) aunque no comparten palabras, lo que confirma que capturan sinónimos, a diferencia de TF-IDF.

Sin embargo, el modelo B también asigna una similitud alta (0.55) a frases no relacionadas, mientras que A la mantiene baja (0.25).

A (Mini LM) separa mejor las frases parecidas de las distintas, y B (MPNET) requerirá un umbral más alto para no responder preguntas fuera de dominio.

Al ser solo 3 frases, luego lo verifico con el dataset de evaluación en la consigna d).

**En este caso, un modelo de embedding mas grande no es necesariamente mejor para este caso particular.**

In [ ]:
# @title
from sklearn.metrics.pairwise import cosine_similarity

m = SentenceTransformer(emb_a)
v = m.encode(['cuanto sale la cuota?', 'que valor tiene la mensualidad?', 'tienen estacionamiento?'])
print('Matriz de siilitud de coseno, emb_a MINI LM')
print(cosine_similarity(v))

In [ ]:
# @title
from sklearn.metrics.pairwise import cosine_similarity

m = SentenceTransformer(emb_b)
v = m.encode(['cuanto sale la cuota?', 'que valor tiene la mensualidad?', 'tienen estacionamiento?'])
print('Matriz de siilitud de coseno, emb_b MPNET')
print(cosine_similarity(v))


## c) Implemente una clase ChatBot usando lo elegido en b).

Puede usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas. Recuerde que sus datos para su BD conocimiento es el dataset que Ud. planteó en el TP3.

• create_collection: crear la "tabla" (en Chroma se llama colección).

• add: guardar los vectores con su texto y su intención.

• query: pasar el vector de una consulta y recibir los más parecidos.

In [ ]:
# @title
cliente = chromadb.Client()   # la base de datos vectorial, vive en la memoria de Colab

class ChatBot:

  def __init__(self, nombre_modelo, umbral):
    self.modelo = SentenceTransformer(nombre_modelo)
    self.umbral = umbral

    # crea la colección-tabla en la base, una por modelo
    '''
    nombre_coleccion, es:
    a = paraphrase-multilingual-MiniLM-L12-v2
    b = paraphrase-multilingual-mpnet-base-v2
    '''
    nombre_coleccion = nombre_modelo.split("/")[-1]

    # get or create, crea o reutiliza la tabla
    self.coleccion = cliente.get_or_create_collection(
      name=nombre_coleccion,
    # que compare con coseno, como en el TP3
      metadata={"hnsw:space": "cosine"})


    # vectorizo las preguntas del TP3 y guardo con su intención
    vectores = self.modelo.encode(preguntas).tolist()
    # utilizo el indice como ID e intencion como metadata
    self.coleccion.add(
      ids=[str(i) for i in range(len(preguntas))],
      embeddings=vectores,
      documents=preguntas,
      metadatas=[{"intencion": i} for i in intenciones])

  def buscar(self, consulta):

    vector = self.modelo.encode([consulta]).tolist()
    resultado = self.coleccion.query(query_embeddings=vector, n_results=3)
    # intención de la más parecida
    intencion = resultado["metadatas"][0][0]["intencion"]
    # Chroma da distancia, la paso a similitud
    similitud = 1 - resultado["distances"][0][0]

    return intencion, similitud, resultado

  def responder(self, consulta):

    intencion, similitud, resultado = self.buscar(consulta)

    # si la similitud no pasa el umbral de 0.5 deriva
    if similitud < self.umbral:
      return "No tengo esa información, te va a responder alguien del equipo.", "fuera_de_dominio", similitud
    # la respuesta del TP3 para esa intención
    contexto = respuestas[intencion]
    mensajes = [
      {"role": "system", "content": "Sos el asistente de WhatsApp de un gimnasio."
       "Respondé en español, breve y amable, maximo de 2 oraciones"
       "usando SOLO la información que te doy. No inventes precios, horarios ni links ni repreguntas: "
       "Antes de responder, revisá si la información responde EXACTAMENTE lo que pregunta el cliente. "
       "Si no lo responde, contestá solo: 'No tengo esa información, te va a responder alguien del equipo.' "
       "Si aparece $$$ o (...), copialo tal cual."},
      {"role": "user", "content": f"Información: {contexto}\n\nPregunta del cliente: {consulta}"}
    ]
    salida = llm(mensajes, max_new_tokens=100, do_sample=False)
    texto = salida[0]["generated_text"][-1]["content"]
    return texto, intencion, similitud

In [ ]:
# creo los dos chatbots: mismo molde, distinto modelo de embeddings
bot_A = ChatBot(emb_a, umbral=0.6)
bot_B = ChatBot(emb_b, umbral=0.6)

print(bot_A.responder("que horarios tienen disponibles, como reservo?"))
print(bot_B.responder('que horarios tienen disponibles?'))

## d) Pruebe el chatbot creado en c) con las preguntas de su dataset a)

Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings.

Justifique y determine cuál elegiría para su aplicación.

In [ ]:
def probar(bot):
  filas = []
  # recorre las listas paralelas
  for pregunta, esperada in zip(preguntas_eval, intenciones_eval):
    # trae el return de la funcion responder
    respuesta, intencion, similitud = bot.responder(pregunta)
    # agrega una fila con las respuestas ordenadas segun el dict clave = columna
    filas.append({
      'pregunta': pregunta,
      'esperada': esperada,
      'detectada': intencion,
      'similitud': round(similitud, 2),
      'correcta': intencion == esperada, # true o false
      'respuesta': respuesta})
  return pd.DataFrame(filas)

res_A = probar(bot_A)
res_B = probar(bot_B)
# tiempo :
pd.set_option('display.max_colwidth', None)   # para ver las respuestas completas
display(res_A)
display(res_B)


## Observaciones sobre las respuestas con intenciones mal detectadas:

#### Error 1: Intencion equivocada (Esto sucede desde Chroma)
#### con un umbral de 0.5
* 3: Rechazada aunque era clara → No existe la palabra *funcional* en la base de preguntas original, solo aparece en las respuestas y Chroma busc en las preguntas
* 14 Estacionamiento → Pasó el umbral por 0.56 en A y 0.59 en B, A la interpreto como cupo_horario y B como info_levantamiento_olimpico. Es necesario subir el valor de umbral
* 16 Vestuarios → en A y B pasó el umbral por una similitud muy alta, muy similar a 'quisiera saber mas sobre el gimnasio' en info_entrenamiento, probablemente la palabra mas importante sea gimnasio y no vestuarios. Se puede reformular la pregunta base.
* 3_b , 2_b, 15, 14 pasaron el umbral por menos de 0.1 puntos, → aumentar el umbral y testear.
* 12 Como me doy de alta, detecta una categoria muy similar inicio_poerlifting / inscripcion

#### umbral 0.6
Con un umbral de 0.6, ambos embedings coinciden en las preguntas problematicas, el modelo B erra en "como me doy de alta?"

El bot se equivoca menos, pero rechaza preguntas validas. Para atención al cliente conviene el umbral más alto: es preferible derivar a una persona que responder mal.
La causa de este fallo es que ninguna de las preguntas de la base contiene las palabras clave de las preguntas en las que existe el fallo.

* Solucion: Ampliar la base de concimiento con consultas que aún no existen y que sean distintas de las preguntas de evaluacion.
* Existen preguntas ambiguas en en inicio_powerlifting e inscripciones que tambien pueden mejorar si amplio la base.

Con respuestas inventadas por el LLM para fuera_de_dominio y respuestas inventadas por el LLM donde pide datos personales: Reforzar las instrucciones del LLM.

#### Instrucciones reforzadas y base de conocimiento ampliada:
Modelo A acierta 16 de 18 (+1)

Modelo B acierta 15 de 18. (+3)

## El modelo de embedding elegido para este caso será el modelo A Mini LM

In [ ]:
print('Aciertos modelo A:', res_A['correcta'].sum(), 'de', len(res_A))
print('Aciertos modelo B:', res_B['correcta'].sum(), 'de', len(res_B))

# solo las que falló cada uno
display(res_A[res_A['correcta'] == False])
display(res_B[res_B['correcta'] == False])

In [ ]:
def metricas_contexto(bot):
  filas = []
  for pregunta, esperada in zip(preguntas_eval, intenciones_eval):
    if esperada == 'fuera_de_dominio':
      continue   # no hay información correcta para recuperar

    intencion, similitud, resultado = bot.buscar(pregunta)
    recuperadas = [m['intencion'] for m in resultado['metadatas'][0]]   # intenciones de las 3
    correctas = recuperadas.count(esperada)

    filas.append({
      'pregunta': pregunta,
      'esperada': esperada,
      'recuperadas': recuperadas,
      'context_precision': round(correctas / 3, 2),
      'context_recall': 1 if correctas > 0 else 0})
  return pd.DataFrame(filas)

ctx = metricas_contexto(bot_A)

display(ctx)


print('Modelo -> precision:', round(ctx['context_precision'].mean(), 2),
      '| recall:', round(ctx['context_recall'].mean(), 2))


## Medicion de tiempos:
El chatabot con el el modelo A de embedding, es mas chico, ha tenido menos errores en el acierto de intenciones, y ademas es mas velóz respondiendo.

El tiempo de respuesta real lo determina el LLM, en este caso es de 13 segundos

In [ ]:
import time

def medir_busqueda(bot):
  tiempos = []
  for pregunta in preguntas_eval:
    inicio = time.time()
    bot.buscar(pregunta)
    tiempos.append(time.time() - inicio)
  return sum(tiempos) / len(tiempos)

# una búsqueda de "calentamiento" antes de medir (la primera siempre es más lenta)
bot_A.buscar("hola")
bot_B.buscar("hola")

t_A = medir_busqueda(bot_A)
t_B = medir_busqueda(bot_B)

print(f'Modelo A: {t_A * 1000:.1f} milisegundos por búsqueda')
print(f'Modelo B: {t_B * 1000:.1f} milisegundos por búsqueda')

inicio = time.time()
bot_A.responder("cuanto sale la cuota?")
print('Respuesta completa:', time.time() - inicio, 'segundos')

# Conclusiones:


## Del TP3 al TP5.
El chatbot pasó de devolver una respuesta fija por intención a un esquema RAG: busca la información en una base de datos vectorial (Chroma) y un LLM (Qwen2.5-1.5B-Instruct) redacta la respuesta.

Las respuestas son más naturales y pueden adaptarse a la pregunta, a cambio de más tiempo de cómputo (unos 12 segundos por respuesta, frente a los 13 milisegundos de la búsqueda).

## Elección del modelo de embeddings.
Elijo el modelo A (MiniLM). Acertó 16 de 18 preguntas frente a 15 de 18 del modelo B (mpnet), su búsqueda es más del doble de rápida (13 ms frente a 31 ms) y separa mejor las frases parecidas de las no relacionadas, como mostró la matriz de similitud. Un modelo más grande no fue mejor para este problema.

## Umbral:
Con un umbral de 0.6 se rechazaron las 4 preguntas fuera de dominio, pero también 2 preguntas válidas. Para atención al cliente prefiero este equilibrio: es mejor derivar a una persona que responder algo incorrecto.

## Tipos de errores:

Encontré dos:
`Errores de búsqueda:` preguntas que usan palabras ausentes en la base ("actividades", "funcional") e intenciones parecidas que se confunden (inscripción y powerlifting). Se corrigen ampliando y revisando la base de conocimiento, sin modificar el dataset de evaluación.
`Errores de generación:` cuando la información recuperada no correspondía, el LLM inventó datos (pasos de registro, vestuarios). Reforzar las instrucciones lo redujo, pero el prompt más estricto a veces hace que el modelo rechace preguntas que sí podía responder. Un modelo de 1.5B tiene límites para seguir instrucciones.

## Métricas de contexto:
El modelo A obtuvo context precision 0.72 y recall 0.93; el modelo B, precision __ y recall __. El recall alto indica que la información correcta casi siempre llega entre los 3 resultados: en "que actividades tienen?" la búsqueda la encontró y el rechazo se debió al umbral. La precision menor indica que el LLM también recibe información de otros temas, lo que aumenta el riesgo de que mezcle o invente.

## Mejoras posibles (para el negocio real):
Ampliar la base con más preguntas reales de clientes, separar las consultas de horarios fijos de las de cupos, consultar los cupos en tiempo real desde la app de turnos del gimnasio e implementar el chatbot en WhatsApp con un LLM por API, que sigue mejor las instrucciones.